In [1]:
"""Using the requests and BeautifulSoup libraries,
   scrape all books listed across at least 3 different book categories (or, if you prefer, the first 5 paginated listing pages of the "All products" catalogue — either scope is acceptable as long as your final dataset has at least 60 books).
   For each book capture: title, price (as listed, in GBP), star_rating (as text, e.g. "Three"),
   availability (as listed text), and category.
"""

'Using the\xa0requests\xa0and\xa0BeautifulSoup\xa0libraries,\n   scrape\xa0all books listed across at least 3 different book categories\xa0(or, if you prefer, the first 5 paginated listing pages of the "All products" catalogue — either scope is acceptable as long as your final dataset has\xa0at least 60 books).\n   For each book capture:\xa0title,\xa0price\xa0(as listed, in GBP),\xa0star_rating\xa0(as text, e.g. "Three"),\n \xa0 availability\xa0(as listed text), and\xa0category.\n'

In [2]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

# Website URL
BASE_URL = "https://books.toscrape.com/"

# Three categories with their actual URLs
categories = {
    "Travel": "https://books.toscrape.com/catalogue/category/books/travel_2/index.html",
    "Mystery": "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html"
}

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_books = []


# Function to scrape one category
def scrape_category(category, url):

    while url:

        print("Scraping:", category, url)

        # Send request
        response = requests.get(url, headers=headers)

        # Check whether request was successful
        print("Status code:", response.status_code)

        # Parse HTML
        soup = BeautifulSoup(response.text, "html.parser")

        # Find all books
        books = soup.find_all("article", class_="product_pod")

        print("Books found:", len(books))

        # Extract information from each book
        for book in books:

            # Title
            title = book.h3.a["title"]

            # Price
            price = book.find(
                "p",
                class_="price_color"
            ).text.strip()

            # Star rating
            rating = book.find(
                "p",
                class_="star-rating"
            )["class"]

            star_rating = rating[1]

            # Availability
            availability = book.find(
                "p",
                class_="instock availability"
            ).text.strip()

            # Store data
            all_books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category
            })

        # Check for next page
        next_button = soup.find("li", class_="next")

        if next_button:

            next_page = next_button.a["href"]

            url = urljoin(url, next_page)

        else:

            url = None


# Scrape all three categories
for category, url in categories.items():

    scrape_category(category, url)


# Convert to DataFrame
df = pd.DataFrame(all_books)


# Display results
print("\n-----------------------------")
print("SCRAPING COMPLETED")
print("-----------------------------")

print("Total books:", len(df))

print("\nFirst 10 records:")
print(df.head(10))


# Save to CSV
df.to_csv(
    "books_three_categories.csv",
    index=False
)

print("\nCSV file saved successfully!")

Scraping: Travel https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Status code: 200
Books found: 11
Scraping: Mystery https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Status code: 200
Books found: 20
Scraping: Mystery https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
Status code: 200
Books found: 12
Scraping: Historical Fiction https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Status code: 200
Books found: 20
Scraping: Historical Fiction https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html
Status code: 200
Books found: 6

-----------------------------
SCRAPING COMPLETED
-----------------------------
Total books: 69

First 10 records:
                                               title    price star_rating  \
0                            It's Only the Himalayas  Â£45.17         Two   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...  Â£49.43       

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         69 non-null     object
 1   price         69 non-null     object
 2   star_rating   69 non-null     object
 3   availability  69 non-null     object
 4   category      69 non-null     object
dtypes: object(5)
memory usage: 2.8+ KB


In [4]:
df.describe()

,title,price,star_rating,availability,category
count,69,69,69,69,69
unique,69,69,5,1,3
top,It's Only the Himalayas,Â£45.17,Three,In stock,Mystery
freq,1,1,16,69,32


In [5]:
df

,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel
...,...,...,...,...,...
64,While You Were Mine,Â£41.32,Five,In stock,Historical Fiction
65,The Secret Healer,Â£34.56,Three,In stock,Historical Fiction
66,Starlark,Â£25.83,Three,In stock,Historical Fiction
67,Lost Among the Living,Â£27.70,Four,In stock,Historical Fiction


In [6]:
"""2. Clean the scraped fields into proper types:
    - Strip the currency symbol from `price` and convert it to a `float` column `price_gbp`.
    - Convert the text star rating (`One`…`Five`) into an integer column `rating` (1–5).
    - Parse the `availability` text into a boolean column `in_stock`.
    - If any field fails to parse for a given row (e.g., unexpected text), handle it with the median-imputation approach for numeric fields or drop the row (state and justify your choice) — do not leave the pipeline crashing on messy rows.
"""

'2. Clean the scraped fields into proper types:\n    - Strip the currency symbol from\xa0`price`\xa0and convert it to a\xa0`float`\xa0column\xa0`price_gbp`.\n    - Convert the text star rating (`One`…`Five`) into an integer column\xa0`rating`\xa0(1–5).\n    - Parse the\xa0`availability`\xa0text into a boolean column\xa0`in_stock`.\n    - If any field fails to parse for a given row (e.g., unexpected text), handle it with the median-imputation approach for numeric fields or drop the row (state and justify your choice) — do not leave the pipeline crashing on messy rows.\n'

In [7]:
df["price_gbp"] = (
    df["price"]
    .str.replace("Â£", "", regex=False)
    .astype(float)
)


In [8]:
n_bad_price = df["price_gbp"].isna().sum()
if n_bad_price:
    median_price = df["price_gbp"].median()
    df["price_gbp"] = df["price_gbp"].fillna(median_price)
    print(f"{n_bad_price} unparseable price(s) imputed with median £{median_price:.2f}")

In [9]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)

In [10]:
n_bad_rating = df["rating"].isna().sum()
if n_bad_rating:
    df = df.dropna(subset=["rating"]).reset_index(drop=True)
    print(f"Dropped {n_bad_rating} row(s) with an unrecognized star rating")

In [11]:
df["in_stock"] = df["availability"].str.strip().eq("In stock")

In [12]:
"""Convert price_gbp to a price_inr column using the project's fixed baseline conversion rate: 1 GBP = 105.50 INR.
This is an artificial, project-defined constant for this assignment,
not a live or historical market rate, so it never needs a lookup or a date reference. This fixed-rate conversion is the required, keyless baseline and is what gets graded for this task — it requires no external API call and no network access; simply state this exact rate in your README
"""

"Convert\xa0price_gbp\xa0to a\xa0price_inr\xa0column using the project's fixed baseline conversion rate:\xa01 GBP = 105.50 INR.\nThis is an artificial, project-defined constant for this assignment,\nnot a live or historical market rate, so it never needs a lookup or a date reference. This fixed-rate conversion is the\xa0required, keyless baseline and is what gets graded\xa0for this task — it requires no external API call and no network access; simply state this exact rate in your README\n"

In [13]:
GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR
df

,title,price,star_rating,availability,category,price_gbp,rating,in_stock,price_inr
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel,45.17,2,True,4765.435
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel,49.43,4,True,5214.865
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel,48.87,3,True,5155.785
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel,36.94,2,True,3897.170
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel,37.33,3,True,3938.315
...,...,...,...,...,...,...,...,...,...
64,While You Were Mine,Â£41.32,Five,In stock,Historical Fiction,41.32,5,True,4359.260
65,The Secret Healer,Â£34.56,Three,In stock,Historical Fiction,34.56,3,True,3646.080
66,Starlark,Â£25.83,Three,In stock,Historical Fiction,25.83,3,True,2725.065
67,Lost Among the Living,Â£27.70,Four,In stock,Historical Fiction,27.70,4,True,2922.350


In [14]:
df.isnull().sum()

,0
title,0
price,0
star_rating,0
availability,0
category,0
price_gbp,0
rating,0
in_stock,0
price_inr,0


In [15]:
"""Using Python's sqlite3 (or pandas.DataFrame.to_sql),
insert your cleaned, converted data into this schema.
Then write and execute at least 5 SQL queries against the database that collectively demonstrate: SELECT/WHERE, ORDER BY, LIMIT, DISTINCT, and (IN or BETWEEN) — plus at least one JOIN between your two tables
(e.g., "list the 10 highest-rated books per category").
Save each query string and its output. """

'Using Python\'s\xa0sqlite3\xa0(or\xa0pandas.DataFrame.to_sql),\ninsert your cleaned, converted data into this schema.\nThen write and execute\xa0at least 5 SQL queries\xa0against the database that collectively demonstrate:\xa0SELECT/WHERE,\xa0ORDER BY,\xa0LIMIT,\xa0DISTINCT, and (IN\xa0or\xa0BETWEEN) — plus\xa0at least one\xa0JOIN\xa0between your two tables\n(e.g., "list the 10 highest-rated books per category").\nSave each query string and its output. '

In [16]:
import os
if os.path.exists("books.db"):
    os.remove("books.db")

In [17]:
import sqlite3

# Create SQLite database
conn = sqlite3.connect("books.db")

cursor = conn.cursor()

# -----------------------------
# Categories table
# -----------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
)
""")

# -----------------------------
# Books table
# -----------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

In [18]:
# Get the unique categories from the DataFrame
categories = df["category"].dropna().unique()

# Insert each category into the categories table
for category in categories:
    cursor.execute(
        "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
        (category,)
    )

conn.commit()

print("Categories inserted successfully!")

Categories inserted successfully!


In [19]:
cursor.execute("SELECT * FROM categories")

categories_data = cursor.fetchall()

for row in categories_data:
    print(row)

(1, 'Travel')
(2, 'Mystery')
(3, 'Historical Fiction')


In [20]:
category_map = {}

cursor.execute("SELECT category_id, category_name FROM categories")

for category_id, category_name in cursor.fetchall():
    category_map[category_name] = category_id

print(category_map)

{'Travel': 1, 'Mystery': 2, 'Historical Fiction': 3}


In [21]:
# Insert every book from the DataFrame
for book_id, row in df.reset_index(drop=True).iterrows():

    category_id = category_map[row["category"]]

    cursor.execute("""
        INSERT INTO books (
            book_id,
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        book_id + 1,
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        int(row["in_stock"]),
        category_id
    ))

conn.commit()

print("Books inserted successfully!")

Books inserted successfully!


In [22]:
cursor.execute("SELECT * FROM books LIMIT 10")

books_data = cursor.fetchall()

for row in books_data:
    print(row)

(1, "It's Only the Himalayas", 45.17, 4765.435, 2, 1, 1)
(2, 'Full Moon over Noahâ\x80\x99s Ark: An Odyssey to Mount Ararat and Beyond', 49.43, 5214.865, 4, 1, 1)
(3, 'See America: A Celebration of Our National Parks & Treasured Sites', 48.87, 5155.785, 3, 1, 1)
(4, 'Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel', 36.94, 3897.1699999999996, 2, 1, 1)
(5, 'Under the Tuscan Sun', 37.33, 3938.3149999999996, 3, 1, 1)
(6, 'A Summer In Europe', 44.34, 4677.870000000001, 2, 1, 1)
(7, 'The Great Railway Bazaar', 30.54, 3221.97, 1, 1, 1)
(8, 'A Year in Provence (Provence #1)', 56.88, 6000.84, 4, 1, 1)
(9, 'The Road to Little Dribbling: Adventures of an American in Britain (Notes From a Small Island #2)', 23.21, 2448.655, 1, 1, 1)
(10, 'Neither Here nor There: Travels in Europe', 38.95, 4109.225, 3, 1, 1)


In [23]:
cursor.execute("SELECT COUNT(*) FROM books")
book_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM categories")
category_count = cursor.fetchone()[0]

print("Number of books:", book_count)
print("Number of categories:", category_count)

Number of books: 69
Number of categories: 3


In [24]:
queries = {

    # 1. SELECT + WHERE
    "Q1_SELECT_WHERE": """
        SELECT title, price_gbp, price_inr
        FROM books
        WHERE price_gbp > 40
    """,

    # 2. ORDER BY
    "Q2_ORDER_BY": """
        SELECT title, rating
        FROM books
        ORDER BY rating DESC
    """,

    # 3. LIMIT
    "Q3_LIMIT": """
        SELECT title, price_gbp
        FROM books
        ORDER BY price_gbp DESC
        LIMIT 10
    """,

    # 4. DISTINCT
    "Q4_DISTINCT": """
        SELECT DISTINCT category_name
        FROM categories
    """,

    # 5. BETWEEN
    "Q5_BETWEEN": """
        SELECT title, price_gbp, rating
        FROM books
        WHERE rating BETWEEN 3 AND 5
        ORDER BY rating DESC
    """,

    # 6. JOIN + ORDER BY + LIMIT
    "Q6_JOIN": """
        SELECT
            b.title,
            c.category_name,
            b.rating,
            b.price_gbp,
            b.price_inr
        FROM books AS b
        JOIN categories AS c
            ON b.category_id = c.category_id
        ORDER BY b.rating DESC, b.price_gbp DESC
        LIMIT 10
    """
}

In [25]:
query_results = {}

with open("sql_query_results.txt", "w", encoding="utf-8") as file:

    for query_name, query in queries.items():

        print("\n" + "=" * 60)
        print(query_name)
        print("=" * 60)

        # Execute query
        cursor.execute(query)

        # Get column names
        columns = [description[0] for description in cursor.description]

        # Get rows
        rows = cursor.fetchall()

        # Convert output to DataFrame
        result_df = pd.DataFrame(rows, columns=columns)

        # Store result in dictionary
        query_results[query_name] = result_df

        # Display query
        print("\nSQL Query:")
        print(query.strip())

        # Display output
        print("\nOutput:")
        print(result_df.to_string(index=False))

        # Save query and output to file
        file.write("\n" + "=" * 60 + "\n")
        file.write(query_name + "\n")
        file.write("=" * 60 + "\n")

        file.write("\nSQL Query:\n")
        file.write(query.strip() + "\n")

        file.write("\nOutput:\n")
        file.write(result_df.to_string(index=False))
        file.write("\n\n")

print("\nAll queries executed successfully!")
print("Saved to: sql_query_results.txt")


Q1_SELECT_WHERE

SQL Query:
SELECT title, price_gbp, price_inr
        FROM books
        WHERE price_gbp > 40

Output:
                                                                   title  price_gbp  price_inr
                                                 It's Only the Himalayas      45.17   4765.435
      Full Moon over Noahâs Ark: An Odyssey to Mount Ararat and Beyond      49.43   5214.865
      See America: A Celebration of Our National Parks & Treasured Sites      48.87   5155.785
                                                      A Summer In Europe      44.34   4677.870
                                        A Year in Provence (Provence #1)      56.88   6000.840
                                                           Sharp Objects      47.82   5045.010
                                                     The Past Never Ends      56.50   5960.750
                         The Murder of Roger Ackroyd (Hercule Poirot #4)      44.10   4652.550
                        

In [26]:
"""Read back at least two of the above query results into pandas DataFrames using
 pd.read_sql(...), and separately reproduce the join-query's result using pd.merge(...) directly on your in-memory DataFrames (no SQL)
 show that both approaches produce equivalent output."""

"Read back\xa0at least two\xa0of the above query results into pandas DataFrames using\n\xa0pd.read_sql(...), and separately reproduce the join-query's result using\xa0pd.merge(...)\xa0directly on your in-memory DataFrames (no SQL)\n show that both approaches produce equivalent output."

In [27]:
# Read Query 1 result from SQLite
q1_df = pd.read_sql(
    queries["Q1_SELECT_WHERE"],
    conn
)

# Read Query 6 JOIN result from SQLite
q6_sql_df = pd.read_sql(
    queries["Q6_JOIN"],
    conn
)

print("Query 1 result:")
display(q1_df.head(10))

print("\nJOIN query result using pd.read_sql():")
display(q6_sql_df)

Query 1 result:


,title,price_gbp,price_inr
0,It's Only the Himalayas,45.17,4765.435
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.865
2,See America: A Celebration of Our National Par...,48.87,5155.785
3,A Summer In Europe,44.34,4677.870
4,A Year in Provence (Provence #1),56.88,6000.840
5,Sharp Objects,47.82,5045.010
6,The Past Never Ends,56.50,5960.750
7,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.550
8,The Last Mile (Amos Decker #2),54.21,5719.155
9,A Time of Torment (Charlie Parker #14),48.35,5100.925



JOIN query result using pd.read_sql():


,title,category_name,rating,price_gbp,price_inr
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,55.53,5858.415
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,5,52.30,5517.650
2,A Time of Torment (Charlie Parker #14),Mystery,5,48.35,5100.925
3,While You Were Mine,Historical Fiction,5,41.32,4359.260
4,The Red Tent,Historical Fiction,5,35.66,3762.130
5,Mrs. Houdini,Historical Fiction,5,30.25,3191.375
6,The Passion of Dolssa,Historical Fiction,5,28.32,2987.760
7,"1,000 Places to See Before You Die",Travel,5,26.08,2751.440
8,What Happened on Beale Street (Secrets of the ...,Mystery,5,25.37,2676.535
9,The Silkworm (Cormoran Strike #2),Mystery,5,23.05,2431.775


In [28]:
# Create the books DataFrame in memory
books_mem = df.reset_index(drop=True).copy()

# Create the same book_id values used in the SQLite database
books_mem["book_id"] = books_mem.index + 1


# Create categories DataFrame directly from the original DataFrame
unique_categories = sorted(
    df["category"].dropna().unique()
)

categories_mem = pd.DataFrame({
    "category_id": range(1, len(unique_categories) + 1),
    "category_name": unique_categories
})

print("Books in-memory DataFrame:")
display(books_mem.head())

print("\nCategories in-memory DataFrame:")
display(categories_mem)

Books in-memory DataFrame:


,title,price,star_rating,availability,category,price_gbp,rating,in_stock,price_inr,book_id
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel,45.17,2,True,4765.435,1
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel,49.43,4,True,5214.865,2
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel,48.87,3,True,5155.785,3
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel,36.94,2,True,3897.170,4
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel,37.33,3,True,3938.315,5



Categories in-memory DataFrame:


,category_id,category_name
0,1,Historical Fiction
1,2,Mystery
2,3,Travel


In [29]:
# Reproduce the JOIN using Pandas only
q6_merge_df = (
    books_mem
    .merge(
        categories_mem,
        left_on="category",
        right_on="category_name",
        how="inner"
    )
    .sort_values(
        by=["rating", "price_gbp"],
        ascending=[False, False]
    )
    [
        [
            "title",
            "category_name",
            "rating",
            "price_gbp",
            "price_inr"
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

print("JOIN result using pd.merge():")
display(q6_merge_df)

JOIN result using pd.merge():


,title,category_name,rating,price_gbp,price_inr
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,55.53,5858.415
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,5,52.30,5517.650
2,A Time of Torment (Charlie Parker #14),Mystery,5,48.35,5100.925
3,While You Were Mine,Historical Fiction,5,41.32,4359.260
4,The Red Tent,Historical Fiction,5,35.66,3762.130
5,Mrs. Houdini,Historical Fiction,5,30.25,3191.375
6,The Passion of Dolssa,Historical Fiction,5,28.32,2987.760
7,"1,000 Places to See Before You Die",Travel,5,26.08,2751.440
8,What Happened on Beale Street (Secrets of the ...,Mystery,5,25.37,2676.535
9,The Silkworm (Cormoran Strike #2),Mystery,5,23.05,2431.775


In [30]:
# Make sure both DataFrames have the same row indexes
q6_sql_df = q6_sql_df.reset_index(drop=True)
q6_merge_df = q6_merge_df.reset_index(drop=True)


# Compare the two results
are_equal = q6_sql_df.equals(q6_merge_df)


# Put SQL and Pandas outputs side by side
comparison_df = pd.concat(
    [
        q6_sql_df.add_prefix("SQL_"),
        q6_merge_df.add_prefix("PANDAS_")
    ],
    axis=1
)


print("SQL JOIN result vs Pandas pd.merge() result:")
display(comparison_df)


print("\nDo both approaches produce equivalent output?")
print(are_equal)

SQL JOIN result vs Pandas pd.merge() result:


,SQL_title,SQL_category_name,SQL_rating,SQL_price_gbp,SQL_price_inr,PANDAS_title,PANDAS_category_name,PANDAS_rating,PANDAS_price_gbp,PANDAS_price_inr
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,55.53,5858.415,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,55.53,5858.415
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,5,52.30,5517.650,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,5,52.30,5517.650
2,A Time of Torment (Charlie Parker #14),Mystery,5,48.35,5100.925,A Time of Torment (Charlie Parker #14),Mystery,5,48.35,5100.925
3,While You Were Mine,Historical Fiction,5,41.32,4359.260,While You Were Mine,Historical Fiction,5,41.32,4359.260
4,The Red Tent,Historical Fiction,5,35.66,3762.130,The Red Tent,Historical Fiction,5,35.66,3762.130
5,Mrs. Houdini,Historical Fiction,5,30.25,3191.375,Mrs. Houdini,Historical Fiction,5,30.25,3191.375
6,The Passion of Dolssa,Historical Fiction,5,28.32,2987.760,The Passion of Dolssa,Historical Fiction,5,28.32,2987.760
7,"1,000 Places to See Before You Die",Travel,5,26.08,2751.440,"1,000 Places to See Before You Die",Travel,5,26.08,2751.440
8,What Happened on Beale Street (Secrets of the ...,Mystery,5,25.37,2676.535,What Happened on Beale Street (Secrets of the ...,Mystery,5,25.37,2676.535
9,The Silkworm (Cormoran Strike #2),Mystery,5,23.05,2431.775,The Silkworm (Cormoran Strike #2),Mystery,5,23.05,2431.775



Do both approaches produce equivalent output?
True
